#Cell 1

# 📧 Spam Mail Detector

## Objective

Build a machine learning classifier that can distinguish between:

- **Spam** — unwanted or fraudulent messages
- **Ham** — legitimate messages

## Dataset

We will use the SMS Spam Collection dataset.

## Machine Learning Model

We will use:

- Text preprocessing
- TF-IDF feature extraction
- Multinomial Naive Bayes classification

## Evaluation

We will measure:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

Finally, we will test the model using our own custom messages.

In [ ]:
#Cell 2

!pip -q install pandas numpy nltk scikit-learn matplotlib seaborn joblib

In [ ]:
#Cell 3

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

In [ ]:
#Cell 3

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

print("Number of English stopwords:", len(stop_words))

In [ ]:
#Cell 5

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_csv(
    url,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset loaded successfully!")
print("Number of rows:", len(df))

In [ ]:
#Cell 6:View the Dataset

df.head(10)

In [ ]:
#Cell 6: Check dataset information

print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns)

print("\nDataset information:")
df.info()

In [ ]:
#Cell 7:Check for missing values

print("Missing values:")
print(df.isnull().sum())

In [ ]:
#Cell 9:Check spam and ham counts

print(df["label"].value_counts())

In [ ]:
#Cell 10:Visualize spam VS ham

In [ ]:
#Cell 11:Examine Example Messages

print("Example HAM messages:")
print()

for message in df[df["label"] == "ham"]["message"].head(5):
    print("-", message)

print("\n" + "=" * 80)

print("\nExample SPAM messages:")
print()

for message in df[df["label"] == "spam"]["message"].head(5):
    print("-", message)

In [ ]:
#Cell 12:Create the text preprocessing function

def clean_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove numbers, punctuation and special characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Split sentence into individual words
    words = text.split()

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Apply stemming
    words = [
        stemmer.stem(word)
        for word in words
    ]

    # Join words back into a sentence
    return " ".join(words)

In [ ]:
#Cell 13:Test the preprcessing Function

example = "Congratulations! You have won $1,000. Claim your prize now!"

cleaned = clean_text(example)

print("Original:")
print(example)

print("\nCleaned:")
print(cleaned)

In [ ]:
#cell 14:Apply preprocessing to the entire dataset

df["clean_message"] = df["message"].apply(clean_text)

df.head(10)

In [ ]:
#cell 15:Comparing Original and cleaned messages

comparison = df[["message", "clean_message"]].head(10)

comparison

In [ ]:
#cell:Prepare X and Y

X = df["clean_message"]
y = df["label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
#cell 17:Split into testing and training data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
#Cell 18:Create TF-IDF Vectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

In [ ]:
#cell 19:Inspect the TF-IDF Vocabulary

feature_names = vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))

print("\nFirst 30 features:")
print(feature_names[:30])

In [ ]:
#cell 20:Train the Naive Bayes models

model = MultinomialNB()

model.fit(
    X_train_tfidf,
    y_train
)

print("Model trained successfully!")

In [ ]:
#cell 21:Make predictions

y_pred = model.predict(X_test_tfidf)

print("Predictions completed!")
print("\nFirst 20 predictions:")
print(y_pred[:20])

In [ ]:
#cell 22:Calculate Accuracy

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
#cell 22:Calculate precision,recall and f1

precision = precision_score(
    y_test,
    y_pred,
    pos_label="spam"
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label="spam"
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label="spam"
)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

In [ ]:
#cell 24:Classifiction report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Ham", "Spam"]
    )
)

In [ ]:
#cel 25:Create confusion matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["ham", "spam"]
)

print(cm)

In [ ]:
#cell 26:Visualize confusion matrix

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Ham", "Spam"],
    yticklabels=["Ham", "Spam"]
)

plt.title("Spam Mail Detector - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

In [ ]:
#cell 27:find incorrectly classified messages

results = pd.DataFrame({
    "message": df.loc[X_test.index, "message"],
    "actual": y_test,
    "predicted": y_pred
})

wrong_predictions = results[
    results["actual"] != results["predicted"]
]

print("Number of incorrect predictions:", len(wrong_predictions))

wrong_predictions.head(20)

In [ ]:
#cell 28:custom message prediction function

def predict_message(message):

    # Clean the message
    cleaned_message = clean_text(message)

    # Convert the message into TF-IDF features
    message_features = vectorizer.transform(
        [cleaned_message]
    )

    # Make prediction
    prediction = model.predict(
        message_features
    )[0]

    # Get probabilities
    probabilities = model.predict_proba(
        message_features
    )[0]

    # Get class names
    classes = model.classes_

    # Find probability for predicted class
    predicted_index = list(classes).index(prediction)
    confidence = probabilities[predicted_index]

    return prediction, confidence

In [ ]:
#cell 29:Custom message predict function

def predict_message(message):

    # Clean the message
    cleaned_message = clean_text(message)

    # Convert the message into TF-IDF features
    message_features = vectorizer.transform(
        [cleaned_message]
    )

    # Make prediction
    prediction = model.predict(
        message_features
    )[0]

    # Get probabilities
    probabilities = model.predict_proba(
        message_features
    )[0]

    # Get class names
    classes = model.classes_

    # Find probability for predicted class
    predicted_index = list(classes).index(prediction)
    confidence = probabilities[predicted_index]

    return prediction, confidence

In [ ]:
#cell 30:Test custom message

custom_messages = [
    "Congratulations! You have won a free iPhone. Claim your prize now!",
    "Hey bro, are you coming to class tomorrow?",
    "URGENT! You have won $5000. Call now to claim your reward.",
    "Please remember to send me the assignment before 6pm.",
    "FREE entry to win cash prizes. Text WIN to 12345 now!",
    "Can you pick me up after school?"
]

for message in custom_messages:

    prediction, confidence = predict_message(message)

    print("Message:")
    print(message)

    print("\nPrediction:")
    print(prediction.upper())

    print("Confidence:")
    print(f"{confidence:.2%}")

    print("-" * 80)

In [ ]:
#cell 30:Test one message interactively

user_message = input("Enter a message to classify: ")

prediction, confidence = predict_message(
    user_message
)

print("\nPrediction:", prediction.upper())
print("Confidence:", f"{confidence:.2%}")

In [ ]:
#cell 31:Save the trained model

joblib.dump(
    model,
    "spam_detector_model.pkl"
)

joblib.dump(
    vectorizer,
    "tfidf_vectorizer.pkl"
)

print("Model saved successfully!")
print("Vectorizer saved successfully!")


In [ ]:
#cell 32:Download the trained files

from google.colab import files

files.download("spam_detector_model.pkl")
files.download("tfidf_vectorizer.pkl")

#Cell 33:Final Project Summary

# 🎯 Project Summary

We successfully built a Spam Mail Detector using machine learning.

## What we did

1. Loaded the SMS Spam Collection dataset.
2. Explored the dataset.
3. Checked for missing values.
4. Visualized spam and ham messages.
5. Cleaned the text.
6. Removed stopwords.
7. Applied stemming.
8. Split the dataset into training and testing sets.
9. Converted text into TF-IDF features.
10. Trained a Multinomial Naive Bayes classifier.
11. Made predictions.
12. Evaluated the model using:
    - Accuracy
    - Precision
    - Recall
    - F1-score
13. Created a confusion matrix.
14. Investigated incorrect predictions.
15. Tested custom messages.
16. Saved the trained model.

## Machine Learning Pipeline

Text
→ Cleaning
→ TF-IDF
→ Naive Bayes
→ Prediction
→ Evaluation

## Skills Learned

- Natural Language Processing
- Text preprocessing
- Stopword removal
- Stemming
- TF-IDF
- Classification
- Naive Bayes
- Model evaluation
- Confusion matrices
- Making predictions with a trained model